In [ ]:
# Import necessary libraries
import numpy as np

# Define function to calculate IBIT strikes based on BTC price
def calculate_ibit_strikes(btc_price, strike_offset=1750):
    """
    Calculate the IBIT strike price based on the BTC price.

    :param btc_price: Current BTC price
    :param strike_offset: IBIT's multiplier for BTC (default: 1750)
    :return: Calculated IBIT strike
    """
    return btc_price / strike_offset

# Function to calculate the Kalshi implied probability based on BTC price
def calculate_kalshi_probability(btc_price, strike_price, direction='YES'):
    """
    Calculate the probability based on Kalshi contract parameters.

    :param btc_price: Current BTC price
    :param strike_price: The strike price for Kalshi contract (e.g., 100000)
    :param direction: Direction for Kalshi ('YES' or 'NO')
    :return: Implied probability for the Kalshi contract
    """
    if direction == 'YES':
        probability = btc_price / strike_price
    elif direction == 'NO':
        probability = 1 - (btc_price / strike_price)
    else:
        raise ValueError("Direction must be 'YES' or 'NO'")

    return probability

# Function to determine the arbitrage conditions
def arb_decision(btc_price, strike_price, direction, ibit_spot):
    """
    Calculate which IBIT strikes would be required to fully arb based on the given input values.

    :param btc_price: Current BTC price
    :param strike_price: The strike price for Kalshi contract
    :param direction: Direction for Kalshi ('YES' or 'NO')
    :param ibit_spot: The current IBIT price (derived from BTC spot)
    :return: Required IBIT strikes for full arbitrage
    """
    # Calculate IBIT price based on BTC price
    ibit_strike = calculate_ibit_strikes(btc_price, btc_price/ibit_spot)

    # Check if the current IBIT spot is higher or lower than the Kalshi target strike
    kalshi_prob = calculate_kalshi_probability(btc_price, strike_price, direction)

    if direction == 'YES':
        # When BTC is above the strike price (Kalshi YES contract)
        print(f"Kalshi probability for YES (BTC > {strike_price}): {kalshi_prob:.2f}")
        print(f"Suggested IBIT strike (based on BTC {btc_price}): {ibit_strike:.2f}")
    elif direction == 'NO':
        # When BTC is below the strike price (Kalshi NO contract)
        print(f"Kalshi probability for NO (BTC < {strike_price}): {kalshi_prob:.2f}")
        print(f"Suggested IBIT strike (based on BTC {btc_price}): {ibit_strike:.2f}")

    # Calculate the potential mispricing for arb
    if kalshi_prob > 0.5:  # For YES (when probability is greater than X%)
        print("Possible arbitrage: IBIT strike undervalued compared to Kalshi YES")
    else:
        print("Possible arbitrage: IBIT strike overvalued compared to Kalshi NO")

    return ibit_strike, kalshi_prob

# Example usage:
btc_price = 100000  # Example BTC spot price
ibit_spot = 50.0  # Example IBIT spot price
strike_price = 100000  # Example Kalshi strike (e.g., BTC > Xk)

# Kalshi contract direction ('YES' or 'NO')
direction = 'YES'
# Kalshi contract rules ('above' or 'below')
contract_direction = 'above'

xor_value = (direction == 'NO') != (contract_direction == 'below')
print(xor_value)

# Call the function to determine the arbitrage conditions
arb_strike, kalshi_prob = arb_decision(btc_price, strike_price, direction, ibit_spot)

if xor_value:
    # below + yes
    short_leg = np.floor(arb_strike)
    long_leg = short_leg + 1
else:
    # above + yes
    short_leg = np.ceil(arb_strike)
    long_leg = short_leg + 1

print("Kalshi contract: BTC " + contract_direction + " " + str(strike_price) + " " + direction)
print("IBIT contract: buy " + str(long_leg) + " sell " + str(short_leg))